# 云南省干旱-植被多源数据联合分析（Notebook 版本）

> 本 notebook 在 `yunnan_drought_analysis.py` 基础上，按分块流程扩展 NDVI / EVI / GOSIF / SPEI，并加入机器学习模型。

## 目标
1. 绘制云南省土壤水分与 GPP 的当月去趋势异常时间序列。
2. 识别 2009-2010 干旱最严重区域，并对比该区域 GPP 异常。
3. 融合 NDVI/EVI、GOSIF、SPEI，对干旱与植被异常同步性进行对比。
4. 用机器学习分析 2009-2015 连续小干旱过程对 GPP 异常的影响。

In [ ]:
# 0) 环境与依赖
import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling
from scipy.signal import detrend
from osgeo import gdal
import xarray as xr

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 1) 路径与参数配置（请按本地数据路径修改）
CONFIG = {
    'shapefile': r"E:/大创/省份边界shp/云南省.shp",
    'gpp_dir': r"G:/数据/FluxSat/FluxSat_GPP_2000_2022",
    'gosif_dir': r"G:/数据/Orig",
    'modis_dir': r"G:/modis",
    'sm_nc': r"D:/GLEAM/v3.8a/SMroot_1980-2022_GLEAM_v3.8a_MO.nc",
    'landcover_tif': r"E:/大创/MCD12Q1_IGBP_0p05deg_2010.tif",
    'spei_dir': r"D:/spei",
    'start': '2000-03-01',
    'end': '2022-05-01',
    'event_start': '2009-01-01',
    'event_end': '2010-12-01',
    'ml_start': '2009-01-01',
    'ml_end': '2015-12-01',
}
period = pd.date_range(CONFIG['start'], CONFIG['end'], freq='MS')
shapefile = gpd.read_file(CONFIG['shapefile'])
min_lon, min_lat, max_lon, max_lat = shapefile.total_bounds
print('时间长度:', len(period), '个月')
print('云南范围:', (min_lon, min_lat, max_lon, max_lat))

In [ ]:
# 2) 通用函数：当月异常 + 去趋势

def compute_monthly_detrended_anomaly(series: pd.Series) -> pd.DataFrame:
    # 1) 距平（按月份气候态） 2) 对距平序列去趋势
    df = pd.DataFrame({'value': series}).copy()
    monthly_mean = df['value'].groupby(df.index.month).transform('mean')
    df['anomaly'] = df['value'] - monthly_mean
    valid = df['anomaly'].notna()
    df['anomaly_detrended'] = np.nan
    df.loc[valid, 'anomaly_detrended'] = detrend(df.loc[valid, 'anomaly'].values)
    return df


def read_masked_raster_mean(raster_path: str, geo_df: gpd.GeoDataFrame) -> float:
    with rasterio.open(raster_path) as src:
        out_img, _ = mask(src, [geo_df.geometry.unary_union], crop=True)
        arr = out_img[0].astype(float)
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan
        arr[~np.isfinite(arr)] = np.nan
        return float(np.nanmean(arr))

In [ ]:
# 3) 读取 GPP、SM 省域平均并计算去趋势异常

gpp_values = []
for t in period:
    fn = Path(CONFIG['gpp_dir']) / f"{t.year:04d}_{t.month:02d}_FluxSat.tif"
    if not fn.exists():
        gpp_values.append(np.nan)
        continue
    gpp_values.append(read_masked_raster_mean(str(fn), shapefile))

gpp_df = compute_monthly_detrended_anomaly(pd.Series(gpp_values, index=period, name='gpp'))

sm_ds = xr.open_dataset(CONFIG['sm_nc'])
sm_data = sm_ds['SMroot'].sel(time=slice(CONFIG['start'], CONFIG['end']))
sm_data = sm_data.sel(lon=slice(min_lon, max_lon), lat=slice(max_lat, min_lat))
sm_mean = sm_data.mean(dim=['lat', 'lon'], skipna=True).to_pandas()
sm_mean.index = pd.to_datetime(sm_mean.index)
sm_df = compute_monthly_detrended_anomaly(sm_mean)

print('GPP与SM序列准备完成')
display(gpp_df.head(3))
display(sm_df.head(3))

# 时间序列图（有坐标和单位）
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(sm_df.index, sm_df['anomaly_detrended'], color='tab:blue', lw=1.5)
axes[0].axhline(0, color='k', ls='--', lw=0.8)
axes[0].set_ylabel('去趋势土壤水分异常 (m³/m³)')
axes[0].set_title('云南省土壤水分去趋势异常时间序列')
axes[0].grid(alpha=0.3)

axes[1].plot(gpp_df.index, gpp_df['anomaly_detrended'], color='tab:green', lw=1.5)
axes[1].axhline(0, color='k', ls='--', lw=0.8)
axes[1].set_xlabel('时间')
axes[1].set_ylabel('去趋势 GPP 异常 (g C m⁻² month⁻¹)')
axes[1].set_title('云南省 GPP 去趋势异常时间序列')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 4) 读取 NDVI / EVI（MOD13C2 HDF）并计算异常
ndvi_data_list, evi_data_list = [], []
os.chdir(CONFIG['modis_dir'])

for year in range(2000, 2023):
    for month in range(1, 13):
        if year == 2000 and month < 3:
            continue
        if year == 2022 and month > 5:
            continue
        filename = f"MOD13C2.A{year:04d}.M{month:02d}.hdf"
        if not os.path.exists(filename):
            continue

        ndvi_ds = gdal.Open(f'HDF4_EOS:EOS_GRID:"{filename}":MOD_Grid_monthly_CMG_VI:CMG 0.05 Deg Monthly NDVI')
        evi_ds = gdal.Open(f'HDF4_EOS:EOS_GRID:"{filename}":MOD_Grid_monthly_CMG_VI:CMG 0.05 Deg Monthly EVI')
        if ndvi_ds is None or evi_ds is None:
            continue

        ndvi = ndvi_ds.ReadAsArray().astype(float)
        evi = evi_ds.ReadAsArray().astype(float)
        ndvi[ndvi == -3000] = np.nan
        evi[evi == -3000] = np.nan
        ndvi *= 0.0001
        evi *= 0.0001

        ndvi_data_list.append([pd.Timestamp(f'{year}-{month:02d}-01'), float(np.nanmean(ndvi))])
        evi_data_list.append([pd.Timestamp(f'{year}-{month:02d}-01'), float(np.nanmean(evi))])

ndvi_df = pd.DataFrame(ndvi_data_list, columns=['time', 'ndvi']).set_index('time')
evi_df = pd.DataFrame(evi_data_list, columns=['time', 'evi']).set_index('time')
ndvi_df = compute_monthly_detrended_anomaly(ndvi_df['ndvi'])
evi_df = compute_monthly_detrended_anomaly(evi_df['evi'])
print('NDVI/EVI完成:', ndvi_df.shape, evi_df.shape)

In [ ]:
# 5) 读取 GOSIF 并计算异常
gosif_data = []
os.chdir(CONFIG['gosif_dir'])

for year in range(2000, 2023):
    for month in range(1, 13):
        if year == 2000 and month < 3:
            continue
        if year == 2022 and month > 5:
            continue
        filename = f"GOSIF_{year:04d}.M{month:02d}.tif"
        if not os.path.exists(filename):
            continue
        with rasterio.open(filename) as src:
            out_img, _ = mask(src, [shapefile.geometry.unary_union], crop=True)
            gosif = out_img[0].astype(float)
            if src.nodata is not None:
                gosif[gosif == src.nodata] = np.nan
            gosif[~np.isfinite(gosif)] = np.nan
            gosif_data.append([pd.Timestamp(f'{year}-{month:02d}-01'), float(np.nanmean(gosif))])

gosif_df = pd.DataFrame(gosif_data, columns=['time', 'gosif']).set_index('time')
gosif_df = compute_monthly_detrended_anomaly(gosif_df['gosif'])
print('GOSIF完成:', gosif_df.shape)

In [ ]:
# 6) 多源对比：同步性
merged = pd.DataFrame(index=period)
merged['SM_anom_dt'] = sm_df['anomaly_detrended']
merged['GPP_anom_dt'] = gpp_df['anomaly_detrended']
merged['NDVI_anom_dt'] = ndvi_df['anomaly_detrended'].reindex(period)
merged['EVI_anom_dt'] = evi_df['anomaly_detrended'].reindex(period)
merged['GOSIF_anom_dt'] = gosif_df['anomaly_detrended'].reindex(period)

corr = merged.corr(method='pearson')
display(corr)

plt.figure(figsize=(9, 7))
sns.heatmap(corr, cmap='RdBu_r', center=0, annot=True, fmt='.2f')
plt.title('多源去趋势异常相关矩阵（同步性）')
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 6))
for col in merged.columns:
    z = (merged[col] - merged[col].mean()) / merged[col].std()
    plt.plot(merged.index, z, lw=1, label=col)
plt.axhline(0, color='k', ls='--', lw=0.8)
plt.xlabel('时间')
plt.ylabel('标准化去趋势异常 (z-score, 无量纲)')
plt.title('云南省干旱与植被异常同步性对比')
plt.legend(ncol=3, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 7) SPEI 月尺度子图（按你给的逻辑）
def plot_spei_subplot(ax, filename, title, min_lon, max_lon, min_lat, max_lat):
    data = xr.open_dataset(filename)
    lat_values = data['lat'].values
    lat_slice = slice(min_lat, max_lat) if lat_values[0] < lat_values[-1] else slice(max_lat, min_lat)
    spei_data = data['spei'].sel(lat=lat_slice, lon=slice(min_lon, max_lon))
    yunnan_data = spei_data.sel(time=slice('2000-01-01', '2022-12-31'))
    yunnan_mean = yunnan_data.mean(dim=['lat', 'lon'])

    t = pd.to_datetime(yunnan_mean.time.values)
    ax.plot(t, yunnan_mean.values, marker='o', markersize=1, color='#6b5458', linestyle='-', linewidth=0.7)
    for i, tt in enumerate(t):
        if tt.month in [3, 4, 5]:
            ax.scatter(tt, yunnan_mean.values[i], color='#a72126', zorder=5, s=12)
    ax.axhline(y=-0.5, color='r', linestyle='--')
    ax.set_xlabel('Time')
    ax.set_ylabel(title)

fig, axs = plt.subplots(4, 3, figsize=(18, 12))
for i, ax in enumerate(axs.flat):
    month_num = i + 1
    filename = Path(CONFIG['spei_dir']) / f"spei{month_num:02d}.nc"
    if filename.exists():
        plot_spei_subplot(ax, str(filename), f'SPEI{month_num:02d}', min_lon, max_lon, min_lat, max_lat)
    else:
        ax.set_title(f'SPEI{month_num:02d} missing')
        ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 8) 构建月尺度 SPEI 序列并融合
def load_monthly_spei_series(spei_dir, min_lon, max_lon, min_lat, max_lat):
    series_list = []
    for m in range(1, 13):
        fn = Path(spei_dir) / f"spei{m:02d}.nc"
        if not fn.exists():
            continue
        ds = xr.open_dataset(fn)
        lat_values = ds['lat'].values
        lat_slice = slice(min_lat, max_lat) if lat_values[0] < lat_values[-1] else slice(max_lat, min_lat)
        s = ds['spei'].sel(lat=lat_slice, lon=slice(min_lon, max_lon)).mean(dim=['lat', 'lon']).to_pandas()
        s.index = pd.to_datetime(s.index)
        s = s[s.index.month == m]
        series_list.append(s)
    if len(series_list) == 0:
        return pd.Series(dtype=float)
    spei = pd.concat(series_list).sort_index()
    spei.name = 'spei'
    return spei

spei_series = load_monthly_spei_series(CONFIG['spei_dir'], min_lon, max_lon, min_lat, max_lat)
if len(spei_series) > 0:
    spei_df = compute_monthly_detrended_anomaly(spei_series)
    merged['SPEI_anom_dt'] = spei_df['anomaly_detrended'].reindex(period)
else:
    merged['SPEI_anom_dt'] = np.nan

display(merged.tail())

In [ ]:
# 9) 机器学习：2009-2015 连续小干旱对 GPP 的影响
# 小干旱：SPEI < -0.5
# 连续小干旱：至少连续2个月 SPEI < -0.5

def mark_consecutive_drought(spei: pd.Series, threshold=-0.5, min_len=2):
    flag = (spei < threshold).astype(int)
    grp = (flag != flag.shift(1)).cumsum()
    runlen = flag.groupby(grp).transform('sum')
    return ((flag == 1) & (runlen >= min_len)).astype(int)

ml_df = merged.loc[CONFIG['ml_start']:CONFIG['ml_end']].copy()
ml_df['drought_small'] = (ml_df['SPEI_anom_dt'] < -0.5).astype(float)
ml_df['drought_consecutive'] = mark_consecutive_drought(ml_df['SPEI_anom_dt'])

# 加入滞后特征（1~3个月），考虑植被的延迟响应
for lag in [1, 2, 3]:
    for col in ['SM_anom_dt', 'NDVI_anom_dt', 'EVI_anom_dt', 'GOSIF_anom_dt', 'SPEI_anom_dt']:
        ml_df[f'{col}_lag{lag}'] = ml_df[col].shift(lag)

y = ml_df['GPP_anom_dt']
X = ml_df.drop(columns=['GPP_anom_dt']).dropna(axis=1, how='all')

tscv = TimeSeriesSplit(n_splits=5)
models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'RandomForest': RandomForestRegressor(n_estimators=300, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'SVR': SVR(C=1.0, epsilon=0.1)
}

results = []
for name, model in models.items():
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    neg_mse_scores = cross_val_score(pipe, X, y, cv=tscv, scoring='neg_mean_squared_error')
    rmse = np.mean(np.sqrt(-neg_mse_scores))
    results.append([name, rmse])

result_df = pd.DataFrame(results, columns=['model', 'cv_rmse']).sort_values('cv_rmse')
display(result_df)

best_name = result_df.iloc[0]['model']
best_model = models[best_name]
best_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', best_model)
])
best_pipe.fit(X, y)
pred = best_pipe.predict(X)

print(f'最佳模型: {best_name}')
print('训练期 R2 =', round(r2_score(y, pred), 3))
print('训练期 RMSE =', round(np.sqrt(mean_squared_error(y, pred)), 4))

In [ ]:
# 10) 连续小干旱影响可视化与统计
impact_df = ml_df[['GPP_anom_dt', 'drought_consecutive']].dropna().copy()
stats = impact_df.groupby('drought_consecutive')['GPP_anom_dt'].agg(['mean', 'median', 'std', 'count'])
print('连续小干旱(1) vs 非连续小干旱(0) 的 GPP 异常统计：')
display(stats)

plt.figure(figsize=(7, 5))
sns.boxplot(data=impact_df, x='drought_consecutive', y='GPP_anom_dt')
plt.xlabel('是否连续小干旱（SPEI<-0.5且连续≥2个月）')
plt.ylabel('GPP 去趋势异常 (g C m⁻² month⁻¹)')
plt.title('连续小干旱期间 GPP 异常分布（2009-2015）')
plt.tight_layout()
plt.show()

In [ ]:
# 11) 保存关键输出
out_dir = Path('outputs_notebook')
out_dir.mkdir(exist_ok=True)
merged.to_csv(out_dir / 'merged_multisource_anomaly_timeseries.csv', encoding='utf-8-sig')
result_df.to_csv(out_dir / 'ml_model_selection_2009_2015.csv', index=False, encoding='utf-8-sig')
print('已保存:')
for f in out_dir.glob('*'):
    print('-', f)